# Day 6 · 실습 1. GPT의 다음 토큰 예측과 자기회귀 텍스트 생성

**실습 목표:** 사전학습된 GPT가 **다음 토큰의 점수를 계산하고 → 토큰을 선택하고 → 입력에 추가해 다시 예측하는 과정**을 코드로 이해합니다.

**학습 흐름:** A형에서 한 번의 토큰 예측과 입력 갱신을 두 차례 확인 → B형에서 같은 연산을 반복문으로 연결하여 자기회귀 생성 함수 완성

이번 실습에서는 `skt/kogpt2-base-v2` 모델을 사용합니다. **모델의 가중치를 학습하지 않고**, 이미 학습된 GPT가 토큰을 생성하는 과정을 살펴봅니다.

- **A형 | 제공 코드:** 코드 셀 4개를 순서대로 실행하며 Logits, 확률, 선택된 Token ID, 입력 갱신을 확인합니다.
- **B형 | TODO 학생 작성:** 이번에는 문제 **1개**만 진행합니다. A형에서 배운 핵심 연산 3곳만 직접 완성하며, 반복문과 종료 처리는 제공됩니다.
- **실행 환경:** 첫 실행 시 모델과 Tokenizer가 자동 다운로드됩니다. GPU가 있으면 GPU를 사용하고, 없으면 CPU에서 실행합니다. 별도의 데이터셋은 필요하지 않습니다.

## A형 · GPT의 다음 토큰 예측과 자기회귀 생성 이해하기

### A-1. KoGPT2 모델과 Prompt 준비

GPT는 입력 문장을 Tokenizer로 분리한 뒤 Token ID로 변환하여 처리합니다. 이번 셀에서는 한국어 GPT 모델을 불러오고, **입력 문장 → Token ID → 토큰 조각**의 관계를 확인합니다.

**확인할 결과:** `input_ids`의 Shape는 **`(1, T)`**이며, `T`는 실제 토큰화된 길이입니다. 하나의 토큰이 반드시 하나의 단어에 해당하지는 않습니다.

In [ ]:
# 1. 실행 환경과 사전학습 모델 준비

# 필요한 패키지를 설치합니다. PyTorch는 GPU 서버에 설치된 버전을 사용합니다.
# 모델 가중치와 Tokenizer는 Hugging Face에서 첫 실행 시 자동 다운로드됩니다.
%pip -q install "transformers>=4.46,<5"

from pathlib import Path
import torch
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

# 한국어 GPT-2 사전학습 모델과 파일 저장 경로를 지정합니다.
MODEL_ID = "skt/kogpt2-base-v2"    # 한국어 GPT-2 사전학습 모델
CACHE_DIR = Path("assets/hf_cache") # 내려받은 모델·토크나이저가 저장될 캐시 폴더
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# GPU(CUDA)가 있으면 사용하고, 없으면 CPU에서 실행합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

######## 핵심 1: 학습된 GPT와 Tokenizer를 함께 준비합니다.
# Tokenizer는 Prompt를 Token ID로 변환하고, GPT는 다음 토큰의 점수를 계산합니다.
# KoGPT2 공식 예제의 특수 토큰 문자열을 지정합니다.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_ID,
    bos_token="</s>", eos_token="</s>", unk_token="<unk>",
    pad_token="<pad>", mask_token="<mask>",
    cache_dir=str(CACHE_DIR),
)
model = GPT2LMHeadModel.from_pretrained(MODEL_ID, cache_dir=str(CACHE_DIR))
model = model.to(device)
model.eval()  # 실습은 추론만 수행하므로 가중치를 학습하지 않습니다.

# EOS는 생성 종료를 나타냅니다. 실제 ID는 모델 설정에서 가져옵니다.
EOS_ID = int(model.config.eos_token_id)


# 2. Prompt를 Token ID로 변환

PROMPT = "오늘 날씨가"
######## 핵심 2: 입력 문자열을 정수형 Token ID Tensor로 바꿉니다.
# return_tensors="pt"를 사용하면 (배치 크기, 토큰 수)인 (1, T) Shape가 됩니다.
input_ids = tokenizer.encode(PROMPT, return_tensors="pt").to(device)
tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())


# 3. 입력 데이터와 Shape 확인

print("사용 장치:", device)
print("모델:", MODEL_ID)
print("모델 캐시 폴더:", CACHE_DIR.resolve())
print("Prompt:", PROMPT)
print("Token ID:", input_ids[0].tolist())
print("토큰 조각:", tokens)
print("input_ids Shape:", tuple(input_ids.shape))  # (1, T): 배치 1개, 토큰 T개
print("EOS ID:", EOS_ID, " / 토큰:", repr(tokenizer.convert_ids_to_tokens(EOS_ID)))

### A-2. GPT Forward와 마지막 위치의 Logits 확인

GPT는 입력 시퀀스의 **각 위치에서 다음 토큰의 예측 점수(Logits)**를 계산합니다. 따라서 전체 출력의 Shape는 `(1, T, V)`이고, `V`는 어휘집 크기입니다.

**아래 코드에서 다음 두 가지에 집중하세요.**

1. **전체 Logits:** 입력된 `T`개 위치 각각에 대해 어휘집 크기 `V`만큼 점수가 만들어집니다.
2. **마지막 위치 Logits:** 현재 Prompt 바로 뒤의 토큰을 예측하려면 `outputs.logits[:, -1, :]`로 마지막 위치만 가져옵니다. 결과 Shape는 `(1, V)`입니다.

In [ ]:
# 1. GPT Forward로 각 토큰 위치의 예측 점수 계산

# 학습이 아닌 추론이므로 기울기를 계산하지 않습니다.
with torch.no_grad():
    outputs = model(input_ids, use_cache=False)

# 전체 출력은 (1, T, V): 입력 토큰 T개의 각 위치마다 V개 어휘의 점수가 있습니다.
all_logits = outputs.logits         # (1, T, V): 위치 T개 × 어휘집 크기 V

######## 핵심: 마지막 위치의 Logits가 현재 Prompt 뒤에 올 토큰의 예측 점수입니다.
# [:, -1, :]은 모든 배치에서 마지막 입력 위치의 어휘별 점수만 가져옵니다.
# 그 결과는 (1, V)이며 V는 전체 어휘집 크기입니다.
next_logits = all_logits[:, -1, :] # (1, V): 마지막 위치 = Prompt 다음 토큰 예측


# 2. 전체 출력과 마지막 위치의 Shape 비교

print("전체 Logits Shape:", tuple(all_logits.shape))
print("마지막 위치 Logits Shape:", tuple(next_logits.shape))
print("어휘집 크기:", model.config.vocab_size)
print("왜 마지막 위치인가? → 지금까지 입력한 Prompt 바로 뒤의 토큰을 예측하기 때문")

### A-3. Softmax로 확률을 계산하고 다음 토큰 선택하기

A-2에서 가져온 Logits는 아직 확률이 아닌 **어휘별 예측 점수**입니다. Softmax로 확률분포를 만든 뒤 상위 5개 후보를 확인하고, Greedy 방식으로 **확률이 가장 높은 토큰 하나**를 선택합니다.

**확인할 결과:** `next_probs`는 `(1, V)`, `next_token_id`는 `(1, 1)` Shape이며, 선택한 토큰은 상위 5개 후보 중 1위와 일치합니다.

In [ ]:
# 1. 다음 토큰 후보의 확률 계산

######## 핵심 1: Logits는 예측 점수이고, Softmax를 적용하면 확률분포가 됩니다.
# (1, V)의 마지막 축(V)에 Softmax를 적용하면 모든 후보의 확률 합이 1이 됩니다.
next_probs = torch.softmax(next_logits, dim=-1)  # (1, V), 합은 약 1


# 2. 확률이 높은 후보 5개 확인

# Top-5는 상위 후보를 살펴보기 위한 출력이며, 이 단계에서는 토큰을 생성하지 않습니다.
top_probs, top_ids = torch.topk(next_probs, k=5, dim=-1)
print("다음 토큰 상위 5개 후보")
for rank, (token_id, p) in enumerate(zip(top_ids[0], top_probs[0]), start=1):
    token_piece = tokenizer.convert_ids_to_tokens(int(token_id.item()))
    print(f" {rank}위 | ID={int(token_id.item()):5d} | 토큰 조각={token_piece!r} | 확률={p.item():.4f}")


# 3. Greedy 방식으로 토큰 하나 선택

######## 핵심 2: Argmax는 확률이 가장 높은 토큰의 ID를 선택합니다.
# keepdim=True는 결과 Shape를 (1, 1)로 유지합니다.
# 그래야 다음 셀에서 기존 input_ids (1, T) 뒤에 바로 연결할 수 있습니다.
next_token_id = torch.argmax(next_probs, dim=-1, keepdim=True)

print("\n선택한 다음 토큰 ID:", int(next_token_id.item()))
print("선택한 토큰 조각:", repr(tokenizer.convert_ids_to_tokens(int(next_token_id.item()))))
print("next_token_id Shape:", tuple(next_token_id.shape))
print("Top-5의 1위와 같은가?:", int(top_ids[0, 0]) == int(next_token_id.item()))

### A-4. 선택한 토큰을 입력에 추가하고 다시 예측하기

앞에서 선택한 Token ID를 `torch.cat()`으로 기존 `input_ids`에 추가합니다. 입력 길이는 `(1, T)`에서 `(1, T+1)`이 되고, GPT는 **새로 늘어난 문맥**을 바탕으로 다음 토큰을 다시 예측합니다.

**아래 코드에서 다음 세 가지에 집중하세요.**

1. **입력 갱신:** 방금 선택한 토큰이 다음 예측의 입력에 추가됩니다.
2. **두 번째 예측:** 갱신된 입력의 마지막 위치에서 새로운 Logits와 확률을 계산합니다.
3. **결과 비교:** 첫 번째·두 번째로 선택한 토큰과 확률, 입력 문장과 Shape 변화를 비교합니다. 선택한 토큰이 종료 토큰(EOS)인지도 확인합니다.

In [ ]:
# 1. 선택한 다음 토큰을 기존 입력에 추가

######## 핵심 1: 토큰을 하나 추가하면 입력 길이가 T에서 T+1로 늘어납니다.
# input_ids (1, T)와 next_token_id (1, 1)를 토큰 축으로 이어 붙입니다.
# 이렇게 만들어진 새로운 입력이 다음 예측의 문맥이 됩니다.
updated_ids = torch.cat([input_ids, next_token_id], dim=-1)


# 2. 늘어난 문맥으로 두 번째 토큰 예측

# GPT의 가중치는 바꾸지 않고, 입력 문맥만 바꿔 다시 Forward를 실행합니다.
with torch.no_grad():
    updated_outputs = model(updated_ids, use_cache=False)

######## 핵심 2: 바뀐 문맥의 마지막 위치에서 새로운 예측 확률을 계산합니다.
# 두 번째 예측에서도 A-2의 Logits 추출과 A-3의 Softmax·Argmax를 그대로 사용합니다.
second_logits = updated_outputs.logits[:, -1, :]  # (1, V)
second_probs = torch.softmax(second_logits, dim=-1)
second_token_id = torch.argmax(second_probs, dim=-1, keepdim=True)  # (1, 1)


# 3. 첫 번째와 두 번째 예측 결과 비교

# 선택한 각 토큰의 확률을 출력하면 문맥이 달라진 결과를 확인할 수 있습니다.
first_selected_prob = next_probs[0, next_token_id.item()].item()
second_selected_prob = second_probs[0, second_token_id.item()].item()

print("처음 입력:", repr(tokenizer.decode(input_ids[0].tolist(), skip_special_tokens=True)))
print("토큰 1개를 추가한 입력:", repr(tokenizer.decode(updated_ids[0].tolist(), skip_special_tokens=True)))
print("입력 Shape 변화:", tuple(input_ids.shape), "→", tuple(updated_ids.shape))
print("첫 번째 선택:", repr(tokenizer.convert_ids_to_tokens(next_token_id.item())),
      f"| ID={next_token_id.item()} | 확률={first_selected_prob:.4f}")
print("두 번째 선택:", repr(tokenizer.convert_ids_to_tokens(second_token_id.item())),
      f"| ID={second_token_id.item()} | 확률={second_selected_prob:.4f}")
print("첫 번째 선택이 실제 EOS인가?:", next_token_id.item() == EOS_ID)
print("두 번째 선택이 실제 EOS인가?:", second_token_id.item() == EOS_ID)

# A형에서는 한 번 선택하고 다시 예측하는 과정을 보여주기 위해 두 번만 계산했습니다.
# B형에서는 같은 연산을 반복문으로 연결하며, EOS 종료 조건은 제공 코드로 확인합니다.

### A형 핵심 정리

**현재 입력 → 마지막 위치의 Logits → Softmax → Greedy로 토큰 선택 → 입력에 추가 → 새로운 문맥에서 다시 예측**

A형에서는 이 과정을 두 번 확인했습니다. **선택한 토큰이 다음 입력에 추가된다는 사실**이 자기회귀 생성의 핵심이며, B형에서는 지금까지 배운 연산을 반복문 안에서 연결합니다. 실제 출력 토큰은 단어의 일부일 수 있고, 생성된 문장이 항상 자연스럽지는 않을 수 있습니다.

## B형 · 자기회귀 텍스트 생성 함수 완성하기

**문제:** A형에서 확인한 **마지막 위치의 Logits → Softmax·Greedy 선택 → 선택 토큰을 입력에 추가**하는 과정을 반복해 `greedy_generate()` 함수를 완성하세요.

함수 골격, 모델 실행, 반복문, 생성 결과 출력, **EOS 종료 및 최대 길이 제한은 제공 코드**입니다. **아래 TODO 세 곳만** 직접 구현합니다.

1. **TODO 1:** `outputs.logits`의 마지막 위치에서 `next_logits`를 추출합니다. **입력:** `(1, T, V)` → **출력:** `(1, V)` (A-2 참고)
2. **TODO 2:** Softmax와 `torch.argmax(..., keepdim=True)`로 `next_token_id`를 선택합니다. **입력:** `(1, V)` → **출력:** `(1, 1)` (A-3 참고)
3. **TODO 3:** `torch.cat()`으로 `input_ids`에 선택 토큰을 추가합니다. **입력:** `(1, T)`와 `(1, 1)` → **출력:** `(1, T+1)` (A-4 참고)

`raise NotImplementedError(...)`를 지우고 A형에서 배운 코드를 작성하세요. 새로운 라이브러리나 구현하지 않은 생성 알고리즘은 필요하지 않습니다.

In [ ]:
# B형은 A형에서 배운 핵심 연산 세 부분만 직접 완성합니다.
# 반복문, 모델 호출, 출력, EOS 종료 처리 코드는 미리 제공되어 있습니다.
def greedy_generate(prompt, max_new_tokens=12, stop_token_id=None):
    """Greedy로 토큰을 반복 생성합니다. TODO 세 곳의 핵심 연산만 완성하세요."""

    # 1. 시작 문맥과 종료 조건 준비
    # 실제 생성에서는 EOS로 종료합니다. 종료 조건 테스트에서만 다른 Token ID를 사용할 수 있습니다.
    if stop_token_id is None:
        stop_token_id = EOS_ID

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    prompt_length = input_ids.shape[1]

    # 2. 다음 토큰 예측과 입력 갱신 반복
    # 현재 입력을 GPT에 전달합니다. 추론이므로 기울기는 계산하지 않습니다.
    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids, use_cache=False)

        ######## TODO 1: 마지막 위치에서 다음 토큰의 Logits를 가져오세요. (A-2)
        # 입력: outputs.logits (1, 현재 입력 길이 T, 어휘집 크기 V)
        # 출력: next_logits (1, V)
        # 힌트: outputs.logits[:, -1, :]
        raise NotImplementedError("TODO 1: 마지막 위치의 Logits 추출")

        ######## TODO 2: Softmax로 확률을 구한 뒤 Greedy로 토큰 하나를 선택하세요. (A-3)
        # 입력: next_logits (1, V) → Softmax 결과 next_probs (1, V)
        # 출력: next_token_id (1, 1), 선택한 토큰의 ID
        # 힌트: torch.softmax(..., dim=-1), torch.argmax(..., dim=-1, keepdim=True)
        raise NotImplementedError("TODO 2: Softmax + Greedy 토큰 선택")

        ######## TODO 3: 선택한 토큰을 기존 입력 뒤에 이어 붙이세요. (A-4)
        # 입력: input_ids (1, T), next_token_id (1, 1)
        # 출력: 갱신된 input_ids (1, T+1)
        # 힌트: torch.cat([input_ids, next_token_id], dim=-1)
        raise NotImplementedError("TODO 3: Token ID를 기존 입력에 추가")

        # 3. 생성 결과 확인 및 종료 조건 적용 (제공 코드)
        # 어떤 토큰이 선택되었는지 토큰 ID와 토큰 조각으로 출력합니다.
        token_piece = tokenizer.convert_ids_to_tokens(int(next_token_id.item()))
        print(f"{step+1:02d}번째 | Token ID={int(next_token_id.item())} | 조각={token_piece!r}")

        # 실제 EOS 또는 테스트용 종료 토큰이 선택되면 반복을 중단합니다.
        # EOS 비교와 break는 학생 TODO가 아닌 제공 코드입니다.
        if int(next_token_id.item()) == int(stop_token_id):
            print("→ 종료 토큰이 선택되어 생성을 멈춥니다.")
            break

    return input_ids, prompt_length

### B형 결과 확인: 반복 생성과 종료 조건

B형의 TODO를 완성한 뒤 아래 셀을 실행합니다. **생성 순서·Token ID·토큰 조각**이 출력되고, 마지막에는 새롭게 생성한 토큰 수와 전체 텍스트를 확인합니다.

**확인할 결과:** 실제 EOS 또는 최대 생성 길이에서 반복이 종료됩니다. 검증 전용 종료 ID를 사용해 1개 토큰 생성 후 멈추는지도 확인하지만, 이 ID가 실제 EOS라는 뜻은 아닙니다.

In [ ]:
# 1. 실제 Prompt로 여러 토큰 생성

# TODO를 완성하면 생성한 토큰 조각들이 순서대로 출력됩니다.
print("[실제 생성] Prompt:", PROMPT)
generated_ids, prompt_len = greedy_generate(PROMPT, max_new_tokens=10)
print("\n전체 생성 텍스트:", tokenizer.decode(generated_ids[0].tolist(), skip_special_tokens=True))
print("새롭게 생성한 토큰 수:", generated_ids.shape[1] - prompt_len)


# 2. EOS 종료 방식에 해당하는 조건 검사 (검증용)

# 실제 GPT가 짧은 생성 중 EOS를 선택하지 않을 수도 있습니다.
# A형에서 이미 선택한 첫 토큰을 시험용 종료 ID로 지정하여 반복문이 멈추는지 확인합니다.
# 이 토큰을 실제 EOS로 바꾸는 것이 아니라 종료 조건의 동작만 테스트합니다.
test_stop_id = int(next_token_id.item())
print("\n[종료 테스트] 검증용 종료 Token ID:", test_stop_id)
test_ids, test_prompt_len = greedy_generate(PROMPT, max_new_tokens=5, stop_token_id=test_stop_id)
assert test_ids.shape[1] - test_prompt_len == 1, "검증용 종료 토큰을 생성하면 1회 만에 멈춰야 합니다."
print("조기 종료 테스트: 통과 (1개 생성 후 중단)")


# 3. 최대 생성 길이에서 종료되는지 확인

# 실제 어휘집에 없는 -1을 검증용 종료 ID로 사용하여 3번 생성하는지 확인합니다.
print("\n[길이 제한 테스트] 최대 3개 토큰 생성")
nmax_ids, nmax_prompt_len = greedy_generate(PROMPT, max_new_tokens=3, stop_token_id=-1)
assert nmax_ids.shape[1] - nmax_prompt_len == 3, "최대 생성 길이 3을 확인하세요."
print("최대 길이 종료 테스트: 통과 (3개 생성 후 중단)")

### B형 정답

In [ ]:

# B형 정답. Greedy 방식의 자기회귀 텍스트 생성 함수

def greedy_generate(prompt, max_new_tokens=12, stop_token_id=None):
    """GPT가 다음 토큰을 하나씩 예측하고 입력에 추가하여 문장을 생성합니다."""

    # 1. 입력 문장과 종료 조건 준비

    # 별도의 종료 토큰을 지정하지 않으면 실제 EOS Token ID를 사용합니다.
    if stop_token_id is None:
        stop_token_id = EOS_ID

    # Prompt를 GPT가 처리할 수 있는 Token ID Tensor로 변환합니다.
    # input_ids Shape: (1, T)
    # T는 현재 입력된 토큰의 개수입니다.
    input_ids = tokenizer.encode(
        prompt, return_tensors="pt"
    ).to(device)

    prompt_length = input_ids.shape[1]


    # 2. 다음 토큰 예측과 입력 갱신 반복

    # 한 번의 반복에서 새로운 토큰 하나를 생성합니다.
    # 최대 max_new_tokens개까지 생성하며, EOS가 나오면 중간에 종료합니다.
    for step in range(max_new_tokens):

        # 현재까지의 입력을 GPT에 전달합니다.
        # 이미 학습된 모델을 사용하는 추론이므로 가중치를 갱신하지 않습니다.
        with torch.no_grad():
            outputs = model(input_ids, use_cache=False)


        ######## 핵심 1. 마지막 위치의 Logits 추출 (TODO 1 정답)

        # GPT는 입력된 모든 위치에서 다음 토큰의 예측 점수를 계산합니다.
        # outputs.logits Shape: (1, T, V)
        # T: 현재 입력 길이, V: 어휘집 크기

        # 현재 문장 바로 뒤에 올 토큰을 예측하려면
        # 마지막 입력 위치(-1)의 Logits를 사용해야 합니다.
        next_logits = outputs.logits[:, -1, :]  # (1, V)


        ######## 핵심 2. Softmax와 Greedy 토큰 선택 (TODO 2 정답)

        # Logits는 아직 확률이 아닌 예측 점수입니다.
        # Softmax를 적용해 어휘집의 각 토큰에 대한 확률을 계산합니다.
        next_probs = torch.softmax(next_logits, dim=-1)  # (1, V)

        # Greedy Decoding은 가장 높은 확률의 토큰 하나를 선택합니다.
        # keepdim=True로 (1, 1) Shape를 유지해야 기존 입력에 연결할 수 있습니다.
        next_token_id = torch.argmax(
            next_probs, dim=-1, keepdim=True
        )  # (1, 1)


        ######## 핵심 3. 선택한 토큰을 기존 입력에 추가 (TODO 3 정답)

        # 방금 생성한 토큰을 기존 input_ids의 마지막에 연결합니다.
        # (1, T) + (1, 1) → (1, T+1)
        input_ids = torch.cat(
            [input_ids, next_token_id], dim=-1
        )

        # 이렇게 갱신된 input_ids가 다음 반복에서 GPT의 새로운 입력이 됩니다.
        # 즉, GPT는 자신이 생성한 토큰까지 참고하여 다음 토큰을 예측합니다.


        # 3. 생성 결과 확인 및 종료 조건 (제공 코드)

        # 이번 반복에서 선택한 토큰을 출력합니다.
        token_piece = tokenizer.convert_ids_to_tokens(
            int(next_token_id.item())
        )

        print(
            f"{step+1:02d}번째 | "
            f"Token ID={int(next_token_id.item())} | "
            f"조각={token_piece!r}"
        )

        # 생성한 토큰이 EOS라면 반복을 종료합니다.
        # EOS가 나오지 않으면 최대 생성 길이까지 반복합니다.
        if int(next_token_id.item()) == int(stop_token_id):
            print("→ 종료 토큰이 선택되어 생성을 멈춥니다.")
            break

    # 모든 생성이 끝나면 전체 Token ID와 원래 Prompt 길이를 반환합니다.
    return input_ids, prompt_length


### 실습 마무리

이번 실습에서는 **GPT가 다음 토큰을 하나씩 선택하고 입력에 이어 붙이는 자기회귀 생성 과정**을 코드로 확인했습니다.

**이번 실습에서 반드시 기억해야 할 핵심 내용은 다음과 같습니다.**

1. **마지막 위치의 Logits:** 입력 시퀀스의 여러 위치 중 현재 Prompt **바로 다음 토큰**에 대한 예측은 마지막 위치의 Logits를 사용합니다.
2. **예측과 선택의 구분:** Softmax는 Logits를 확률로 바꾸고, Greedy는 그중 확률이 가장 높은 Token ID를 선택합니다.
3. **입력 갱신과 재예측:** `torch.cat()`으로 선택 토큰을 입력에 추가하면 문맥이 길어지고, 갱신된 문맥에서 다시 다음 토큰을 예측합니다.
4. **반복 생성과 종료:** 동일한 과정을 반복해 문장을 생성하며, 실제 EOS가 선택되거나 설정한 최대 생성 길이에 도달하면 멈춥니다.

**결국 자기회귀 생성의 핵심은 방금 생성한 토큰을 다음 예측의 입력으로 사용하면서 문장을 한 토큰씩 확장한다는 것입니다.**